In [1]:
import os
import shutil
import time

import numpy as np
import rasterio
import requests

from selenium import webdriver
from selenium.webdriver.common.by import By


In [2]:
save_dir = "/home/mohamed-ashraf/Desktop/projects/sat-project/data"

BANDS = ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B09", "B11", "B12"]
CLOUD_THRESHOLD = 20
MIN_VALID_FRACTION = 0.35
MAX_OUTLIER_FRACTION = 0.20
MIN_DATES_AFTER_FILTER = 6
START_INDEX = 210

mapping = {
    1: 3,  # Water -> Water
    2: 4,  # Artificial Bare -> Cement
    3: 2,  # Natural Bare -> Sand
    5: 1,  # Woody -> Greenery
    6: 1,  # Cultivated -> Greenery
    7: 1   # Natural Veg -> Greenery
}

tiles = [
    # "28QDE", "29NMG", "29PKL", "29PNM", "30PTV", "30PWT", "31NFH", 
    "31PGR", "31PGS", "32PLR", "32PLS", "32PPA", "32PQQ", "32PRQ", "33KUT", "33KVU", "33KXV", 
    "33LUC", "33LUD", "33LYE", "33MZM", "33PUQ", "33PWM", "35LMC", "35LNF", "35MNT", "35NLA", 
    "35NRD", "35PNL", "35PNR", "35PRL", "36MWT", "36MWU", "36MZT", 
]


In [3]:
def apply_mapping(label):
    mapped = np.zeros_like(label, dtype=np.uint8)
    for original, new in mapping.items():
        mapped[label == original] = new
    return mapped


def save_mask(input_path, save_path):
    with rasterio.open(input_path) as src:
        img = src.read()
        profile = src.profile

    label = img[0].astype(np.uint8)
    confidence = img[1].astype(np.uint8)
    mapped = apply_mapping(label)

    profile.update(count=2, dtype="uint8")

    with rasterio.open(save_path, "w", **profile) as dst:
        dst.write(mapped, 1)
        dst.write(confidence, 2)

    print(f"Saved mask with confidence: {save_path}")


def summarize_scene(masked_img):
    return np.array([
        np.nanmedian(masked_img[band_idx])
        for band_idx in range(masked_img.shape[0])
    ], dtype=np.float32)


def choose_dates(scene_features):
    features = np.stack(scene_features, axis=0)
    center = np.median(features, axis=0)
    scale = np.median(np.abs(features - center), axis=0)
    scale = np.where(scale < 1e-6, 1.0, scale)
    scores = np.mean(np.abs((features - center) / scale), axis=1)

    keep_n = int(np.ceil((1.0 - MAX_OUTLIER_FRACTION) * len(scene_features)))
    keep_n = max(MIN_DATES_AFTER_FILTER, keep_n)
    keep_n = min(len(scene_features), keep_n)

    keep_idx = np.argsort(scores)[:keep_n]
    return keep_idx, scores


In [ ]:
next_index = START_INDEX

for tile_code in tiles:
    for chip_idx in range(30):
        chip_name = f"{chip_idx:02d}"
        chip_url = f"https://source.coop/radiantearth/landcovernet/landcovernet_af/data/v1.0/2018/{tile_code}/{chip_name}"
        raw_dir = os.path.join(save_dir, f"raw_{chip_name}")
        os.makedirs(raw_dir, exist_ok=True)

        driver = webdriver.Chrome()
        driver.get(chip_url)
        time.sleep(5)

        content_div = None
        retries_left = 25
        while retries_left > 0:
            try:
                content_div = driver.find_element(
                    By.CSS_SELECTOR,
                    "div.rt-Box[style*='max-height: 800px'][style*='overflow: auto']"
                )
                break
            except Exception as exc:
                retries_left -= 1
                if retries_left == 0:
                    driver.quit()
                    raise RuntimeError(
                        f"Failed to load content for {tile_code}/{chip_name}. Last error: {exc}"
                    )
                print(f"Retrying {tile_code}/{chip_name} after load failure: {exc}")
                driver.refresh()
                time.sleep(5 if retries_left > 10 else 6)

        session = requests.Session()
        for cookie in driver.get_cookies():
            session.cookies.set(cookie["name"], cookie["value"])

        def download_file(url, path):
            if os.path.exists(path):
                return
            response = session.get(url, stream=True)
            response.raise_for_status()
            with open(path, "wb") as handle:
                for chunk in response.iter_content(8192):
                    handle.write(chunk)

        last_height = 0
        while True:
            driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight", content_div)
            time.sleep(1)
            new_height = driver.execute_script("return arguments[0].scrollHeight", content_div)
            current_top = driver.execute_script("return arguments[0].scrollTop", content_div)
            if new_height == last_height or current_top + 5 >= new_height - content_div.size["height"]:
                break
            last_height = new_height

        links = content_div.find_elements(By.CSS_SELECTOR, "a[href]")
        if not links:
            print(f"Empty chip {tile_code}/{chip_name}, skipping.")
            driver.quit()
            shutil.rmtree(raw_dir)
            continue

        date_urls = []
        mask_url = None
        mask_name = None

        for anchor in links:
            href = anchor.get_attribute("href")
            fname = anchor.get_attribute("download")

            if not href:
                continue

            tail = href.rstrip("/").split("/")[-1]
            if tail.startswith(f"{tile_code}_{chip_name}_20") and len(tail) == len(f"{tile_code}_{chip_name}_20180103"):
                date_urls.append(href.rstrip("/"))

            if fname and fname.endswith("_LC_10m.tif"):
                mask_url = href
                mask_name = fname

        date_urls = sorted(set(date_urls))
        print(f"{tile_code}/{chip_name}: found {len(date_urls)} date folders")

        if mask_url is None:
            driver.quit()
            shutil.rmtree(raw_dir)
            raise RuntimeError(f"LC mask not found for {tile_code}/{chip_name}")

        os.makedirs(os.path.join(save_dir, "masks"), exist_ok=True)
        os.makedirs(os.path.join(save_dir, "imgs"), exist_ok=True)

        mask_local_path = os.path.join(raw_dir, mask_name)
        final_mask_path = os.path.join(save_dir, f"masks/mask_{next_index:02d}.tif")
        download_file(mask_url, mask_local_path)
        save_mask(mask_local_path, final_mask_path)

        raw_images = []
        masked_images = []
        scene_features = []
        out_profile = None

        for date_url in date_urls:
            print("Processing:", date_url)
            driver.get(date_url)
            time.sleep(2)

            anchors = driver.find_elements(By.CSS_SELECTOR, "a[href]")
            band_files = {}
            cld_file = None

            for anchor in anchors:
                href = anchor.get_attribute("href")
                fname = anchor.get_attribute("download")

                if not href or not fname:
                    continue

                matched_band = False
                for band_name in BANDS:
                    if f"_{band_name}_10m.tif" in fname:
                        local_path = os.path.join(raw_dir, fname)
                        download_file(href, local_path)
                        band_files[band_name] = local_path
                        matched_band = True
                        break

                if matched_band:
                    continue

                if "_CLD_10m.tif" in fname:
                    local_path = os.path.join(raw_dir, fname)
                    download_file(href, local_path)
                    cld_file = local_path

            if len(band_files) != len(BANDS) or cld_file is None:
                print("Skipped incomplete date")
                continue

            bands_stack = []
            for band_name in BANDS:
                with rasterio.open(band_files[band_name]) as src:
                    bands_stack.append(src.read(1).astype(np.float32))
                    if out_profile is None:
                        out_profile = src.profile.copy()

            raw_img = np.stack(bands_stack, axis=0)

            with rasterio.open(cld_file) as src:
                cld = src.read(1)

            cloud_mask = cld >= CLOUD_THRESHOLD
            valid_fraction = 1.0 - float(cloud_mask.mean())
            if valid_fraction < MIN_VALID_FRACTION:
                print(f"Skipped low-valid date: {valid_fraction:.2%} valid pixels")
                continue

            masked_img = raw_img.copy()
            masked_img[:, cloud_mask] = np.nan

            scene_feature = summarize_scene(masked_img)
            if np.isnan(scene_feature).any():
                print("Skipped unstable date after masking")
                continue

            raw_images.append(raw_img)
            masked_images.append(masked_img)
            scene_features.append(scene_feature)

        driver.quit()

        if not masked_images:
            print(f"No valid dates for {tile_code}/{chip_name}, skipping.")
            shutil.rmtree(raw_dir)
            continue

        keep_idx, scores = choose_dates(scene_features)
        masked_stack = np.stack([masked_images[idx] for idx in keep_idx], axis=0)
        raw_stack = np.stack([raw_images[idx] for idx in keep_idx], axis=0)

        median_img = np.nanmedian(masked_stack, axis=0)
        fallback_img = np.median(raw_stack, axis=0)
        median_img = np.where(np.isnan(median_img), fallback_img, median_img).astype(np.float32)

        out_profile.update(count=len(BANDS), dtype="float32")
        final_img_path = os.path.join(save_dir, f"imgs/img_{next_index:02d}.tif")
        with rasterio.open(final_img_path, "w", **out_profile) as dst:
            dst.write(median_img)

        print(f"Kept {len(keep_idx)}/{len(masked_images)} dates after outlier filtering")
        print(f"Worst kept outlier score: {scores[keep_idx].max():.3f}")
        print(f"Saved image: {final_img_path}")
        print(f"Saved mask : {final_mask_path}")

        shutil.rmtree(raw_dir)
        next_index += 1


31PGR/00: found 18 date folders
Saved mask with confidence: /home/mohamed-ashraf/Desktop/projects/sat-project/data/masks/mask_210.tif
Processing: https://source.coop/radiantearth/landcovernet/landcovernet_af/data/v1.0/2018/31PGR/00/31PGR_00_20181003
Processing: https://source.coop/radiantearth/landcovernet/landcovernet_af/data/v1.0/2018/31PGR/00/31PGR_00_20181008
Processing: https://source.coop/radiantearth/landcovernet/landcovernet_af/data/v1.0/2018/31PGR/00/31PGR_00_20181013


In [ ]:
import matplotlib.pyplot as plt

img_path = "data/imgs/img_32.tif"
mask_path = "data/masks/mask_32.tif"

with rasterio.open(img_path) as src:
    img = src.read()

rgb = img[[3, 2, 1]].transpose(1, 2, 0)
rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min())

with rasterio.open(mask_path) as src:
    mask = src.read(1)
    confidence = src.read(2)

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.title("RGB Image")
plt.imshow(rgb)

plt.subplot(1, 3, 2)
plt.title("Mask")
plt.imshow(mask, cmap="jet")

plt.subplot(1, 3, 3)
plt.title("Confidence")
plt.imshow(confidence, cmap="magma", vmin=0, vmax=100)

plt.show()
